# 📈 Time Series Analysis

Complete time series workflow using the classic **Airline Passengers dataset** (1949–1960).
Decompose the series, test stationarity, and build an ARIMA forecast.

**Outline:** Load & plot → Decomposition → ADF stationarity test → Differencing → ACF/PACF → ARIMA forecast

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Setup complete ✅')

## 1. Load & Visualise

In [ ]:
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'], index_col='Month')
df.columns = ['passengers']
print(f'Period: {df.index.min().date()} to {df.index.max().date()}')
print(f'Observations: {len(df)}')
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
df['passengers'].plot(ax=ax, color='steelblue', linewidth=2)
ax.set_title('Monthly Airline Passengers (1949–1960)', fontsize=14, fontweight='bold')
ax.set_ylabel('Passengers (thousands)')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

## 2. Seasonal Decomposition

Using **multiplicative** decomposition because variance grows with the level of the series.

In [ ]:
decomp = seasonal_decompose(df['passengers'], model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
for ax, (label, data, color) in zip(axes, [
    ('Observed', decomp.observed, 'steelblue'),
    ('Trend', decomp.trend, 'darkorange'),
    ('Seasonal', decomp.seasonal, 'green'),
    ('Residual', decomp.resid, 'red'),
]):
    data.plot(ax=ax, color=color)
    ax.set_ylabel(label, fontweight='bold')

plt.suptitle('Multiplicative Seasonal Decomposition', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Stationarity Test (ADF)

In [ ]:
def adf_test(series, label=''):
    result = adfuller(series.dropna())
    pval = result[1]
    print(f'{label}')
    print(f'  ADF Statistic : {result[0]:.4f}')
    print(f'  p-value       : {pval:.4f}')
    print(f'  Stationary    : {pval < 0.05}  (reject H0 if p < 0.05)')
    print()
    return pval < 0.05

adf_test(df['passengers'], 'Original series')

## 4. Differencing to Achieve Stationarity

In [ ]:
log_series = np.log(df['passengers'])
log_diff = log_series.diff().dropna()

adf_test(log_diff, 'Log-differenced series')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
log_series.plot(ax=axes[0], title='Log(Passengers)', color='steelblue')
log_diff.plot(ax=axes[1], title='Log-Differenced (d=1)', color='darkorange')
plt.tight_layout()
plt.show()

## 5. ACF & PACF Plots

Used to identify the **MA order (q)** and **AR order (p)** for ARIMA.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(log_diff, lags=40, ax=axes[0])
plot_pacf(log_diff, lags=40, ax=axes[1])
axes[0].set_title('Autocorrelation (ACF)', fontweight='bold')
axes[1].set_title('Partial Autocorrelation (PACF)', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. ARIMA Model & 24-Month Forecast

In [ ]:
train = log_series[:-24]
test  = log_series[-24:]

model  = ARIMA(train, order=(2, 1, 2))
fitted = model.fit()
print(fitted.summary())

In [ ]:
forecast = fitted.get_forecast(steps=24)
pred = np.exp(forecast.predicted_mean)
ci   = np.exp(forecast.conf_int())
actual = np.exp(log_series)

fig, ax = plt.subplots(figsize=(13, 5))
actual[:-24].plot(ax=ax, label='Training data', color='steelblue', linewidth=2)
actual[-24:].plot(ax=ax, label='Actual (held-out)', color='green', linewidth=2)
pred.plot(ax=ax, label='ARIMA(2,1,2) forecast', color='darkorange', linewidth=2, linestyle='--')
ax.fill_between(ci.index, ci.iloc[:, 0], ci.iloc[:, 1], alpha=0.2, color='darkorange', label='95% CI')
ax.set_title('ARIMA Forecast vs Actual', fontsize=14, fontweight='bold')
ax.set_ylabel('Passengers')
ax.legend()
plt.tight_layout()
plt.show()

## Summary

| Step | Finding |
|------|---------|
| Raw series | Non-stationary (trend + multiplicative seasonality) |
| Decomposition | Clear 12-month cycle; trend explains most variance |
| ADF test | p=0.99 on raw → p<0.01 after log + differencing |
| ARIMA(2,1,2) | Reasonable 24-month forecast with widening CI |

**Next steps:** Try SARIMA for explicit seasonal modelling, or Facebook Prophet for a simpler interface.